This notebook is dedicated to cleaning and merging the three data sources 
needed to construct the analysis panel.

The Median Multiple — our primary predictor — will be constructed by dividing 
Zillow median home values by ACS median household income at the county-year level.

Outcome and mediator variables will be extracted from County Health Rankings 
data and column naming conventions standardized across years.

## CHR Cleaning Tasks
- Load correct sheet ('Select Measure Data' for 2024, 'Ranked Measure Data' for all others)
- Use header=[0,1] for MultiIndex
- Drop state row (index 0, FIPS 6000)
- Standardize first-level column names to title case (2016–2022 → 2023–2024 convention)
- Select target columns (FIPS, County, plus 5 variables)
- Rename second-level column names to consistent standard names
- Add year column
- Stack all years into one long dataframe


In [1]:
# imports

import pandas as pd
import numpy as np
import os


In [36]:
# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

files = os.listdir('../data/raw/county_health_rankings')

chr_frames = []

for f in sorted(files):
    if '2025' in f:
        continue
    file_path = '../data/raw/county_health_rankings/' + f
    sheet = 'Select Measure Data' if '2024' in f else 'Ranked Measure Data'
    year = f[:4]
    df_year = pd.read_excel(file_path, sheet_name=sheet, header=[0,1])
    df_year = df_year.drop(index=0)
    df_year = df_year[df_year[('Unnamed: 0_level_0', 'FIPS')].notna()]
    df_year.columns = [(_col[0].title(), _col[1]) for _col in df_year.columns]
    if '2016' in f or '2017' in f or '2018' in f or '2019' in f:
        mental_health_col = 'Mentally Unhealthy Days'
        preventable_col = 'Preventable Hosp. Rate'
        lbw_col = '% LBW'
    elif '2021' in f or '2022' in f:
        lbw_col = '% Low birthweight' 
        mental_health_col = 'Average Number of Mentally Unhealthy Days'
        preventable_col = 'Preventable Hospitalization Rate'
    else:
        mental_health_col = 'Average Number of Mentally Unhealthy Days'
        preventable_col = 'Preventable Hospitalization Rate'
        lbw_col = '% Low Birthweight'
    # create a selection df with data of interest
    df_col_select = df_year[[('Unnamed: 0_Level_0', 'FIPS'),('Unnamed: 2_Level_0', 'County'),
    ('Premature Death', 'Years of Potential Life Lost Rate'), ('Poor Mental Health Days', mental_health_col),
    ('Preventable Hospital Stays', preventable_col), ('Children In Poverty', '% Children in Poverty'), ('Low Birthweight', lbw_col)]].copy()
    df_col_select.columns = ['fips', 'county', 'premature_death_rate', 
                          'poor_mental_health_days', 'preventable_hosp_rate',
                          'children_in_poverty_pct', 'low_birthweight_pct']
    df_col_select['year'] = f[:4]
    chr_frames.append(df_col_select)
    print(f, '| rows:', df_year.shape[0])
    print(df_col_select.columns.tolist())

chr_panel = pd.concat(chr_frames, ignore_index=True)
print(chr_panel)

2016 County Health Rankings California Data - v3.xls | rows: 58
['fips', 'county', 'premature_death_rate', 'poor_mental_health_days', 'preventable_hosp_rate', 'children_in_poverty_pct', 'low_birthweight_pct', 'year']
2017 County Health Rankings California Data - v2.xls | rows: 58
['fips', 'county', 'premature_death_rate', 'poor_mental_health_days', 'preventable_hosp_rate', 'children_in_poverty_pct', 'low_birthweight_pct', 'year']
2018 County Health Rankings California Data - v3.xls | rows: 58
['fips', 'county', 'premature_death_rate', 'poor_mental_health_days', 'preventable_hosp_rate', 'children_in_poverty_pct', 'low_birthweight_pct', 'year']
2019 County Health Rankings California Data - v1_0.xls | rows: 58
['fips', 'county', 'premature_death_rate', 'poor_mental_health_days', 'preventable_hosp_rate', 'children_in_poverty_pct', 'low_birthweight_pct', 'year']
2020 County Health Rankings California Data - v1_0.xlsx | rows: 58
['fips', 'county', 'premature_death_rate', 'poor_mental_health_

In [28]:
files = os.listdir('../data/raw/county_health_rankings')
for f in sorted(files):
    if '2025' in f:
        continue
    file_path = '../data/raw/county_health_rankings/' + f
    sheet = 'Select Measure Data' if '2024' in f else 'Ranked Measure Data'
    df_temp = pd.read_excel(file_path, sheet_name=sheet, header=[0,1])
    for col in df_temp.columns:
        if 'LBW' in col[1] or 'Low birthweight' in col[1] or 'Low Birthweight' in col[1]:
            if '% ' in col[1] and 'CI' not in col[1] and 'AIAN' not in col[1] and 'Asian' not in col[1] and 'Black' not in col[1] and 'Hispanic' not in col[1] and 'White' not in col[1]:
                print(f[:4], '|', col)
                break

2016 | ('Low birthweight', '% LBW')
2017 | ('Low birthweight', '% LBW')
2018 | ('Low birthweight', '% LBW')
2019 | ('Low birthweight', '% LBW')
2020 | ('Low birthweight', '% Low Birthweight')
2021 | ('Low birthweight', '% Low birthweight')
2022 | ('Low birthweight', '% Low birthweight')
2023 | ('Low Birthweight', '% Low Birthweight')
2024 | ('Low Birthweight', '% Low Birthweight')


In [15]:
# verify first-level column names converted to title case method
print(df_year.columns[:5])

Index([   ('Unnamed: 0_Level_0', 'FIPS'),   ('Unnamed: 1_Level_0', 'State'),
        ('Unnamed: 2_Level_0', 'County'), ('Premature Death', 'Unreliable'),
           ('Premature Death', 'Deaths')],
      dtype='object')


In [20]:
# check if the the second-level colunm names were affected by the title case 

for col in df_year.columns:
    if 'Premature' in col[0]:
        print(col)

('Premature Death', 'Unreliable')
('Premature Death', 'Deaths')
('Premature Death', 'Years of Potential Life Lost Rate')
('Premature Death', '95% CI - Low')
('Premature Death', '95% CI - High')
('Premature Death', 'National Z-Score')
('Premature Death', 'YPLL Rate (Hispanic (all races))')
('Premature Death', 'YPLL Rate (Hispanic (all races)) 95% CI - Low')
('Premature Death', 'YPLL Rate (Hispanic (all races)) 95% CI - High')
('Premature Death', 'YPLL Rate (Hispanic (all races)) Unreliable')
('Premature Death', 'YPLL Rate (Non-Hispanic AIAN)')
('Premature Death', 'YPLL Rate (Non-Hispanic AIAN) 95% CI - Low')
('Premature Death', 'YPLL Rate (Non-Hispanic AIAN) 95% CI - High')
('Premature Death', 'YPLL Rate (Non-Hispanic AIAN) Unreliable')
('Premature Death', 'YPLL Rate (Non-Hispanic Asian)')
('Premature Death', 'YPLL Rate (Non-Hispanic Asian) 95% CI - Low')
('Premature Death', 'YPLL Rate (Non-Hispanic Asian) 95% CI - High')
('Premature Death', 'YPLL Rate (Non-Hispanic Asian) Unreliable')


In [3]:
file_path = '../data/raw/county_health_rankings/2016 County Health Rankings California Data - v3.xls' 

chr_df_2016 = pd.read_excel(file_path, sheet_name='Ranked Measure Data', header=[0,1])

chr_df_2016.head(3)


Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0 Premature death  \
                FIPS              State             County        # Deaths   
0             6000.0         California                NaN        312690.0   
1             6001.0         California            Alameda         12315.0   
2             6003.0         California             Alpine             NaN   

                                                                          \
  Years of Potential Life Lost Rate 95% CI - Low 95% CI - High   Z-Score   
0                            5274.6       5249.4        5299.8       NaN   
1                            4885.7       4766.1        5005.2 -0.850338   
2                               NaN          NaN           NaN       NaN   

  Poor or fair health                                       \
          % Fair/Poor 95% CI - Low 95% CI - High   Z-Score   
0                18.1         17.1          19.1       NaN   
1                13.0         12.8          13.3 -0.826464   
2                15.6         15.3          15.9       NaN   

  Poor physical health days                                       \
  Physically Unhealthy Days 95% CI - Low 95% CI - High   Z-Score   
0                       4.0          3.8           4.2       NaN   
1                       3.4          3.3           3.5 -1.032691   
2                       4.2          4.1           4.2       NaN   

  Poor mental health days                                       \
  Mentally Unhealthy Days 95% CI - Low 95% CI - High   Z-Score   
0                     3.6          3.4           3.8       NaN   
1                     3.3          3.2           3.4 -1.360231   
2                     4.1          4.0           4.2       NaN   

  Low birthweight                                                   \
       Unreliable # Low Birthweight Births # Live Births     % LBW   
0             NaN                 248282.0     3655228.0  6.792517   
1             NaN                  10132.0      139884.0  7.243144   
2             NaN                      NaN           NaN       NaN   

                                       Adult smoking               \
  95% CI - Low 95% CI - High   Z-Score     % Smokers 95% CI - Low   
0          6.8           6.8       NaN          12.8         11.9   
1          7.1           7.4  1.267327          10.6         10.4   
2          NaN           NaN       NaN          15.7         15.4   

                          Adult obesity                                      \
  95% CI - High   Z-Score       % Obese 95% CI - Low 95% CI - High  Z-Score   
0          13.8       NaN     22.547845          NaN           NaN      NaN   
1          10.9 -1.524826     20.000000         18.2          21.9 -1.15082   
2          16.1       NaN     23.700000         17.5          30.6      NaN   

  Food environment index             Physical inactivity               \
  Food Environment Index   Z-Score % Physically Inactive 95% CI - Low   
0                    7.7       NaN             16.670569          NaN   
1                    7.6 -0.505515             14.700000         13.2   
2                    6.1       NaN             17.500000         12.6   

                          Access to exercise opportunities            \
  95% CI - High   Z-Score                    % With Access   Z-Score   
0           NaN       NaN                        93.508853       NaN   
1          16.3 -0.852318                        99.733160 -1.164941   
2          23.8       NaN                        96.510638       NaN   

    Excessive drinking                                       \
  % Excessive Drinking 95% CI - Low 95% CI - High   Z-Score   
0                 17.2         16.1          18.3       NaN   
1                 16.3         16.0          16.7 -1.383533   
2                 17.9         17.5          18.2       NaN   

    Alcohol-impaired driving deaths                                      \
  # Alcohol-Impaired Driving Deaths 

In [4]:
chr_df_2016.tail(3)

Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0 Premature death  \
                 FIPS              State             County        # Deaths   
58             6115.0         California               Yuba           877.0   
59                NaN                NaN                NaN             NaN   
60                NaN                NaN                NaN             NaN   

                                                                           \
   Years of Potential Life Lost Rate 95% CI - Low 95% CI - High   Z-Score   
58                            7894.4       7209.0        8579.8  0.912083   
59                               NaN          NaN           NaN       NaN   
60                               NaN          NaN           NaN       NaN   

   Poor or fair health                                       \
           % Fair/Poor 95% CI - Low 95% CI - High   Z-Score   
58                17.9         17.5          18.4  0.444634   
59                 NaN          NaN           NaN       NaN   
60                 NaN          NaN           NaN       NaN   

   Poor physical health days                                       \
   Physically Unhealthy Days 95% CI - Low 95% CI - High   Z-Score   
58                       4.2          4.0           4.3  0.744317   
59                       NaN          NaN           NaN       NaN   
60                       NaN          NaN           NaN       NaN   

   Poor mental health days                                       \
   Mentally Unhealthy Days 95% CI - Low 95% CI - High   Z-Score   
58                     4.0          3.9           4.1  0.695573   
59                     NaN          NaN           NaN       NaN   
60                     NaN          NaN           NaN       NaN   

   Low birthweight                                                   \
        Unreliable # Low Birthweight Births # Live Births     % LBW   
58             NaN                    528.0        8778.0  6.015038   
59             NaN                      NaN           NaN       NaN   
60             NaN                      NaN           NaN       NaN   

                                        Adult smoking               \
   95% CI - Low 95% CI - High   Z-Score     % Smokers 95% CI - Low   
58          5.5           6.5 -0.424653          15.0         14.5   
59          NaN           NaN       NaN           NaN          NaN   
60          NaN           NaN       NaN           NaN          NaN   

                           Adult obesity                                       \
   95% CI - High   Z-Score       % Obese 95% CI - Low 95% CI - High   Z-Score   
58          15.5  0.960076          25.9         20.3          32.2  0.493977   
59           NaN       NaN           NaN          NaN           NaN       NaN   
60           NaN       NaN           NaN          NaN           NaN       NaN   

   Food environment index             Physical inactivity               \
   Food Environment Index   Z-Score % Physically Inactive 95% CI - Low   
58                    5.7  1.544944                  15.1         11.5   
59                    NaN       NaN                   NaN          NaN   
60                    NaN       NaN                   NaN          NaN   

                           Access to exercise opportunities            \
   95% CI - High   Z-Score                    % With Access   Z-Score   
58          19.4 -0.698941                        77.122861  0.298384   
59           NaN       NaN                              NaN       NaN   
60           NaN       NaN                              NaN       NaN   

     Excessive drinking                                      \
   % Excessive Drinking 95% CI - Low 95% CI - High  Z-Score   
58                 18.8         18.2          19.4  0.50491   
59                  NaN          NaN           NaN      NaN   
60                  NaN          NaN           NaN      NaN   

     Alcohol-impaired driving deaths                     